<img src="Institute-of-Chartered-Foresters-edited.png" width="200">

# Supplementary 7. Integrated inferential summary of Supplementary 1 to 6

**Author:** Jorge Lizarazo  

**journal:** *Forestry: An International Journal of Forest Research*

This notebook is the cross-document synthesis of the six analytical
supplementary files. It does not introduce new analyses; instead it collects
the key inferential statistics (t, z, χ², U, W) and p-values from
Supplementary 1 to 6 into a single dashboard. Data sources:

| Source | Supplementary | Content |
| ------ | ------------- | ------- |
| `outputs/summary/stats/s1_predictor_selection_summary.csv` | S1 | Retained predictor set |
| `data/kmeans_metrics_ref.csv`, `data/kmeans_final_metrics.csv`, `data/cluster_composition_by_essence_k4.csv` | S2 | PCA + K-means clustering |
| `outputs/summary/stats/s3_gam_inference_manuscript.csv` | S3 | GAM inference (\|z\|, p-values, spatial smooth χ², edf) |
| `outputs/summary/stats/s4_species_mannwhitney.csv` | S4 | Between-species Mann–Whitney U with rank-biserial effect size |
| `outputs/{sugar,red}/stats/*.csv` | S5, S6 | XGBoost inference (ROC-AUC CI, McNemar, paired bootstrap Δ AUC, permutation importance, Mann–Whitney on conifer covariates, Wilcoxon on future scenarios) |
| `outputs/summary/stats/integrated_inference_summary.csv` | S5 + S6 synthesis | Long-format test table |

All p-values in this notebook are two-sided; FDR-adjusted q-values use the
Benjamini–Hochberg procedure. Inference for the R-based S3 is listed at the
magnitudes reported in the source manuscript because the mgcv fit is
expensive; the CSV gives the exact table used for Supplementary 7 when the
R notebook is re-executed.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
SUMMARY = Path('outputs') / 'summary' / 'stats'
pd.set_option('display.max_colwidth', 160)


## S7.1 Supplementary 1 – Feature selection

Supplementary 1 combines an initial PCA, a LightGBM feature-importance
screening and an iterative Pearson correlation filter (|r| < 0.9). The
LightGBM classifier discriminates *Acer saccharum* (ES) from *A. rubrum*
(EO); feature importance is measured as cumulative gain over all tree
splits. PCA and LightGBM are descriptive (no hypothesis test). After this
pipeline, ten environmental predictors are retained for the grid-based
models of Supplementary 5 and 6 (the variable `forest` is excluded because
its future dynamics cannot be projected reliably).

In [2]:
pred = pd.read_csv(SUMMARY / 's1_predictor_selection_summary.csv')
pred


,model,predictor_type,predictors
0,Model A,environment-only,"bio15, ph, bio7, bio3, bio31, sand, bio34, bio12, bio23, bio1"
1,Model B,environment + conifer kernel + conifer distance,"bio15, ph, bio7, bio3, bio31, sand, bio34, bio12, bio23, bio1, Picea_mariana_kernel_sigma5, Pinus_banksiana_kernel_sigma5, Abies_balsamea_kernel_sigma5, Pic..."


## S7.2 Supplementary 2 – PCA and K-means clustering

After rebalancing and standardising the inventory, PCA on the ten retained
predictors captures about 90 % of the multivariate variance in six
components; the retained component scores feed K-means. Candidate values of
k are ranked by the Silhouette coefficient (↑), the Calinski–Harabasz
index (↑) and the Davies–Bouldin score (↓). These are **internal
validation indices**, not frequentist tests — K-means does not have a
parametric null. The selected `k = 4` solution maximises Silhouette and CH
and is close to the minimum of DB, and the final cluster composition by
species is read from `data/cluster_composition_by_essence_k4.csv`.

In [3]:
km_scan = pd.read_csv(SUMMARY / 's2_kmeans_metrics.csv')
display(km_scan)
km_final = pd.read_csv(SUMMARY / 's2_kmeans_final.csv')
display(km_final)
comp = pd.read_csv(SUMMARY / 's2_cluster_composition.csv')
display(comp)


,k,n_samples,pca_components,silhouette,calinski_harabasz,davies_bouldin
0,2,1413270,6,0.276339,536977.581154,1.436014
1,3,1413270,6,0.302661,534412.548061,1.321662
2,4,1413270,6,0.346317,557685.440591,1.107469
3,5,1413270,6,0.313357,529978.364294,1.159687
4,6,1413270,6,0.325679,526637.770319,1.068813
5,7,1413270,6,0.314612,519928.438924,1.129933
6,8,1413270,6,0.314622,502188.399391,1.151119
7,9,1413270,6,0.320383,498928.643090,1.151345
8,10,1413270,6,0.324376,499456.388112,1.079867
9,11,1413270,6,0.312632,466284.490630,1.195582


,k,n_samples,pca_components,silhouette,calinski_harabasz,davies_bouldin
0,4,1413270,6,0.346317,557685.440591,1.107469


,cluster_k,EO,ES
0,0,62.94,37.06
1,1,64.60,35.40
2,2,68.63,31.37
3,3,48.49,51.51


## S7.3 Supplementary 3 – GAM inference (binomial, logit link)

Supplementary 3 fits a binomial `bam()` (mgcv) with all environmental
predictors as linear terms plus a 2-D thin-plate spline `s(longitude,
latitude, bs = "tp")`. The table below reports the absolute z-statistics
|z| from `summary(gam_fit)$p.table` and the χ² / effective degrees of
freedom of the spatial smooth from `summary(gam_fit)$s.table`. All
p-values are < 0.0001; the spatial smooth absorbs most of the signal
(edf ≈ 98.65, χ² ≈ 53,500), a pattern that motivates the grid-based SDMs
of Supplementary 5 and 6.

> Statistical note. The table is the one used in the manuscript text
> (Table S2); the helper `_stats_addon.py` re-creates this CSV from the
> manuscript reference values so that Supplementary 7 stays reproducible
> without needing the R kernel at render time. Re-executing the R
> supplementary overwrites the file with the live model output.

In [4]:
gam = pd.read_csv(SUMMARY / 's3_gam_inference_manuscript.csv')
gam


,term,abs_z,p_value,direction
0,forest,93.9,<0.0001,negative on A. rubrum odds
1,elevation,92.9,<0.0001,negative on A. rubrum odds
2,bio1 (annual mean T),15.3,<0.0001,negative on A. rubrum odds
3,bio24 (radiation wettest quarter),11.9,<0.0001,positive on A. rubrum odds
4,bio34 (warm-season moisture),11.9,<0.0001,positive on A. rubrum odds
5,bio8 (T wettest quarter),11.4,<0.0001,positive on A. rubrum odds
6,bio30 (lowest moisture),11.1,<0.0001,positive on A. rubrum odds
7,bio31 (moisture seasonality),10.9,<0.0001,negative on A. rubrum odds
8,bio15 (precipitation seasonality),10.8,<0.0001,negative on A. rubrum odds
9,bio27 (radiation coldest quarter),9.7,<0.0001,negative on A. rubrum odds


## S7.4 Supplementary 4 – Between-species Mann–Whitney U tests

Supplementary 4 compares sugar maple and red maple on the retained bioclim
+ edaphic predictors. Because the distributions are non-normal and sample
sizes are very large, we report the Mann–Whitney U statistic with its
large-sample z-transform, the two-sided p-value, and the rank-biserial
effect size r (positive r ⇒ red maple — `EO` — has larger values than
sugar maple — `ES`). A BH-FDR q-value is added as the test is repeated
across predictors.

In [5]:
s4 = pd.read_csv(SUMMARY / 's4_species_mannwhitney.csv')
s4.round(4)


,variable,median_ES,median_EO,n_ES,n_EO,U,z,p_value,effect_r_rank_biserial,p_value_fdr
0,bio1,2.9426,2.8595,565308,1090725,3.541596e+11,157.2199,0.0,-0.1488,0.0
1,bio15,15.1026,15.9382,565308,1090725,2.608037e+11,-162.8154,0.0,0.1541,0.0
2,bio31,7.5032,8.1820,565308,1090725,2.787156e+11,-101.4113,0.0,0.0960,0.0
3,bio34,770.4377,746.2740,565308,1090725,3.770266e+11,235.6107,0.0,-0.2229,0.0
4,bio23,80.0993,81.7441,565308,1090725,2.710007e+11,-127.8589,0.0,0.1210,0.0
5,bio12,1021.4250,1006.3750,565308,1090725,3.183757e+11,34.5483,0.0,-0.0327,0.0
6,ph,4.7133,4.6000,565308,1090725,3.361337e+11,95.4248,0.0,-0.0903,0.0
7,sand,67.0000,64.0000,565308,1090725,3.145083e+11,21.2904,0.0,-0.0201,0.0


## S7.5 Supplementary 5 and 6 – XGBoost inference

The grid-based sugar and red maple models are compared using:

1. Pooled ROC-AUC on the held-out fold concatenation, with Hanley–McNeil
   95 % CIs (`pooled_auc_{species}.csv`).
2. Paired **Model A vs Model B** tests on the same held-out cells:
   McNemar's test with continuity correction on the 0.5-threshold
   predictions, plus a paired bootstrap on Δ ROC-AUC
   (`model_comparison_tests_{species}.csv`).

In [6]:
rows = []
for sp in ('sugar', 'red'):
    d = Path('outputs') / sp / 'stats'
    rows.append(pd.read_csv(d / f'pooled_auc_{sp}.csv'))
pooled = pd.concat(rows, ignore_index=True)
display(pooled.round(4))

rows = []
for sp in ('sugar', 'red'):
    d = Path('outputs') / sp / 'stats'
    rows.append(pd.read_csv(d / f'model_comparison_tests_{sp}.csv'))
comp = pd.concat(rows, ignore_index=True)
display(comp.round(6))


,species,model,n,n_pos,n_neg,roc_auc,roc_auc_se,roc_auc_lo,roc_auc_hi
0,sugar,Model A,1056,176,880,0.6298,0.0242,0.5824,0.6771
1,sugar,Model B,1056,176,880,0.9231,0.0141,0.8954,0.9508
2,red,Model A,1284,214,1070,0.6846,0.0214,0.6426,0.7265
3,red,Model B,1284,214,1070,0.9479,0.0107,0.9269,0.9689


,species,test,statistic,ci_low,ci_high,p_value,n,method,extra
0,sugar,"Paired bootstrap (Delta ROC-AUC, B - A)",0.294149,0.253731,0.337897,0.0,1056,"paired bootstrap, 2000 resamples",NaN
1,sugar,McNemar (correct/incorrect at threshold 0.5),512.529304,NaN,NaN,0.0,546,chi2-cc,"{""b_A_wrong_B_right"": 538, ""c_A_right_B_wrong"": 8}"
2,red,"Paired bootstrap (Delta ROC-AUC, B - A)",0.263763,0.224586,0.305692,0.0,1284,"paired bootstrap, 2000 resamples",NaN
3,red,McNemar (correct/incorrect at threshold 0.5),624.234949,NaN,NaN,0.0,681,chi2-cc,"{""b_A_wrong_B_right"": 667, ""c_A_right_B_wrong"": 14}"


### S7.5.1 Permutation importance of Model B predictors

`perm_z = permutation_mean / permutation_std` is standardised to a normal
approximation. `perm_p_two_sided` is the two-sided p-value; `perm_p_fdr`
is the BH-adjusted q-value across the 16 predictors of Model B for each
species. Only the top-5 by mean |SHAP| are shown for each species
(full tables in `outputs/{species}/stats/`).

In [7]:
rows = []
for sp in ('sugar', 'red'):
    d = Path('outputs') / sp / 'stats'
    t = pd.read_csv(d / f'feature_importance_tests_{sp}.csv')
    t = t.sort_values('shap_mean_abs', ascending=False).head(5)
    t.insert(0, 'species', sp)
    rows.append(t)
pd.concat(rows, ignore_index=True).round(6)


,species,feature,gain_importance,weight_importance,shap_mean_abs,permutation_mean,permutation_std,perm_z,perm_p_two_sided,perm_p_fdr
0,sugar,Abies_balsamea_distance_km,40.560188,80.0,1.762334,0.016388,0.002491,6.579384,0.000000,0.000000
1,sugar,bio1,7.228444,208.0,1.587553,0.019531,0.001898,10.293046,0.000000,0.000000
2,sugar,Abies_balsamea_kernel_sigma5,6.607225,169.0,1.227708,0.000181,0.000053,3.435783,0.000591,0.002363
3,sugar,bio34,3.476376,139.0,0.606137,0.000390,0.000098,3.970249,0.000072,0.000383
4,sugar,Pinus_banksiana_distance_km,5.080130,39.0,0.565175,0.000000,0.000000,0.500000,0.617075,0.759477
5,red,Abies_balsamea_distance_km,64.870712,68.0,1.852531,0.013979,0.001607,8.700291,0.000000,0.000000
6,red,bio1,7.716213,181.0,1.385933,0.011856,0.001735,6.831785,0.000000,0.000000
7,red,Abies_balsamea_kernel_sigma5,5.221425,213.0,1.332781,0.000345,0.000114,3.031724,0.002432,0.009726
8,red,bio34,4.442581,199.0,0.908858,0.001601,0.000486,3.294983,0.000984,0.005249
9,red,Pinus_banksiana_distance_km,4.750463,48.0,0.697750,0.000000,0.000000,0.500000,0.617075,0.844437


### S7.5.2 Mann–Whitney U for conifer covariates

Presence vs background cells on the six conifer covariates. All p-values
are much smaller than 10⁻⁵⁰ for both species, with rank-biserial |r| > 0.7,
confirming that the three conifer species carry a very strong presence
signal relative to the background — this is the empirical basis for the
"conifer-associated ecological filter" in Model B.

In [8]:
rows = []
for sp in ('sugar', 'red'):
    d = Path('outputs') / sp / 'stats'
    t = pd.read_csv(d / f'conifer_mannwhitney_{sp}.csv')
    rows.append(t)
pd.concat(rows, ignore_index=True).round(4)


,species,covariate,median_presence,median_background,U,z,p_value,effect_r_rank_biserial,n_presence,n_background,p_value_fdr
0,sugar,Picea_mariana_kernel_sigma5,0.5143,0.0642,133387.0,15.1472,0.0,-0.7225,176,880,0.0
1,sugar,Pinus_banksiana_kernel_sigma5,0.3653,0.0144,133108.0,15.0717,0.0,-0.7189,176,880,0.0
2,sugar,Abies_balsamea_kernel_sigma5,0.6943,0.0544,145237.0,18.3555,0.0,-0.8755,176,880,0.0
3,sugar,Picea_mariana_distance_km,0.0000,267.2165,12603.5,-17.5540,0.0,0.8372,176,880,0.0
4,sugar,Pinus_banksiana_distance_km,0.0000,283.4259,12929.0,-17.4658,0.0,0.8330,176,880,0.0
5,sugar,Abies_balsamea_distance_km,0.0000,267.2165,12238.0,-17.6529,0.0,0.8420,176,880,0.0
6,red,Picea_mariana_kernel_sigma5,0.5488,0.0536,204908.0,18.2598,0.0,-0.7897,214,1070,0.0
7,red,Pinus_banksiana_kernel_sigma5,0.4130,0.0132,203688.0,18.0134,0.0,-0.7791,214,1070,0.0
8,red,Abies_balsamea_kernel_sigma5,0.7054,0.0465,218980.0,21.1016,0.0,-0.9127,214,1070,0.0
9,red,Picea_mariana_distance_km,0.0000,270.5361,15503.0,-19.9903,0.0,0.8646,214,1070,0.0


### S7.5.3 Wilcoxon signed-rank on future scenarios

Paired across the 22 future scenarios (CMCC, GFDL, HADGEM, IPSL, MIROC,
NORESM × RCP 4.5 / 8.5 × 2040–2079 / 2060–2099), comparing the
environment-only projection with the static-conifer sensitivity analysis
in Supplementary 5 and 6. Both area change and latitudinal shift are
systematically smaller under the static-conifer constraint (p < 0.01 for
both species and both metrics).

In [9]:
rows = []
for sp in ('sugar', 'red'):
    d = Path('outputs') / sp / 'stats'
    rows.append(pd.read_csv(d / f'future_wilcoxon_{sp}.csv'))
pd.concat(rows, ignore_index=True).round(6)


,species,metric,n_scenarios,median_env_only,median_static_conifer,median_difference,W,p_value
0,sugar,area_change_pct,22,124.455,99.440,22.065,40.0,0.003671
1,sugar,lat_shift_km,22,422.450,244.320,172.925,0.0,0.000000
2,red,area_change_pct,22,93.780,57.175,34.535,3.0,0.000002
3,red,lat_shift_km,22,327.160,181.910,145.260,0.0,0.000000


## S7.6 Integrated inference table

All inferential rows from Supplementary 5 and 6 are stacked in a single
long-format CSV (`integrated_inference_summary.csv`) with one row per
test / statistic. The `supplementary` column maps each row back to its
source document. This is the table intended to accompany the manuscript's
Supplementary Methods.

In [10]:
synth = pd.read_csv(SUMMARY / 'integrated_inference_summary.csv')
synth.head(30)


,supplementary,species,component,statistic,value,ci_low,ci_high,p_value,note
0,S5,sugar,Model A,"ROC-AUC (pooled, 2-fold CV held-out)",0.629771,0.582406,0.677137,NaN,"Hanley-McNeil 95% CI, n=1056"
1,S5,sugar,Model B,"ROC-AUC (pooled, 2-fold CV held-out)",0.923069,0.895376,0.950763,NaN,"Hanley-McNeil 95% CI, n=1056"
2,S5,sugar,Model A vs Model B,"Paired bootstrap (Delta ROC-AUC, B - A)",0.294149,0.253731,0.337897,0.000000e+00,"paired bootstrap, 2000 resamples"
3,S5,sugar,Model A vs Model B,McNemar (correct/incorrect at threshold 0.5),512.529304,NaN,NaN,0.000000e+00,chi2-cc
4,S5,sugar,Abies_balsamea_distance_km,Permutation importance z,6.579384,NaN,NaN,4.723999e-11,BH-FDR q = 3.78e-10; mean |SHAP| = 1.76
5,S5,sugar,Abies_balsamea_kernel_sigma5,Permutation importance z,3.435783,NaN,NaN,5.908441e-04,BH-FDR q = 0.00236; mean |SHAP| = 1.23
6,S5,sugar,Pinus_banksiana_distance_km,Permutation importance z,0.500000,NaN,NaN,6.170751e-01,BH-FDR q = 0.759; mean |SHAP| = 0.565
7,S5,sugar,Pinus_banksiana_kernel_sigma5,Permutation importance z,0.654654,NaN,NaN,5.126908e-01,BH-FDR q = 0.759; mean |SHAP| = 0.437
8,S5,sugar,Picea_mariana_distance_km,Permutation importance z,0.000000,NaN,NaN,1.000000e+00,BH-FDR q = 1; mean |SHAP| = 0.31
9,S5,sugar,Picea_mariana_kernel_sigma5,Permutation importance z,0.333333,NaN,NaN,7.388827e-01,BH-FDR q = 0.788; mean |SHAP| = 0.108


In [11]:
synth['p_value_fmt'] = synth['p_value'].apply(lambda p: '<1e-10' if (pd.notna(p) and p < 1e-10) else (f'{p:.3g}' if pd.notna(p) else ''))
synth[['supplementary','species','component','statistic','value','ci_low','ci_high','p_value_fmt','note']].to_csv(SUMMARY / 'integrated_inference_summary_display.csv', index=False)
synth_display = synth[['supplementary','species','component','statistic','value','ci_low','ci_high','p_value_fmt']].copy()
synth_display = synth_display.round({'value': 4, 'ci_low': 4, 'ci_high': 4})
synth_display


,supplementary,species,component,statistic,value,ci_low,ci_high,p_value_fmt
0,S5,sugar,Model A,"ROC-AUC (pooled, 2-fold CV held-out)",0.6298,0.5824,0.6771,
1,S5,sugar,Model B,"ROC-AUC (pooled, 2-fold CV held-out)",0.9231,0.8954,0.9508,
2,S5,sugar,Model A vs Model B,"Paired bootstrap (Delta ROC-AUC, B - A)",0.2941,0.2537,0.3379,<1e-10
3,S5,sugar,Model A vs Model B,McNemar (correct/incorrect at threshold 0.5),512.5293,NaN,NaN,<1e-10
4,S5,sugar,Abies_balsamea_distance_km,Permutation importance z,6.5794,NaN,NaN,<1e-10
5,S5,sugar,Abies_balsamea_kernel_sigma5,Permutation importance z,3.4358,NaN,NaN,0.000591
6,S5,sugar,Pinus_banksiana_distance_km,Permutation importance z,0.5000,NaN,NaN,0.617
7,S5,sugar,Pinus_banksiana_kernel_sigma5,Permutation importance z,0.6547,NaN,NaN,0.513
8,S5,sugar,Picea_mariana_distance_km,Permutation importance z,0.0000,NaN,NaN,1
9,S5,sugar,Picea_mariana_kernel_sigma5,Permutation importance z,0.3333,NaN,NaN,0.739


## S7.7 Narrative synthesis

* **S1 (feature selection, descriptive).** Ten predictors retained
  (`bio15, ph, bio7, bio3, bio31, sand, bio34, bio12, bio23, bio1`); the
  variable `forest` is dropped from the grid-based models of S5 and S6
  because it cannot be projected forward under future climates.
* **S2 (clustering, internal indices).** Silhouette = 0.346, CH = 557,685,
  DB = 1.107 at k = 4, for N = 1,413,270 stands on 6 PCs retaining
  ~90 % of variance.
* **S3 (GAM inference, p-values).** All tested predictors have
  |z| > 9 and p < 0.0001; the spatial thin-plate smooth dominates with
  edf ≈ 98.65 and χ² ≈ 53,500.
* **S4 (between-species tests).** Mann–Whitney U on bioclim + edaphic
  variables separates the two species on every tested variable
  (all p < 1e-100 at the manuscript sample sizes; rank-biserial |r|
  up to ≈ 0.22).
* **S5 and S6 (grid-based SDM).** ROC-AUC increases from ≈ 0.65 and ≈ 0.70
  (Model A, pooled held-out) to ≈ 0.92 and ≈ 0.93 (Model B) for sugar
  and red maple respectively, with non-overlapping 95 % CIs. The paired
  bootstrap confirms Δ AUC ≈ 0.29 (sugar) and ≈ 0.26 (red), p < 0.001;
  McNemar χ² > 500 with p < 0.001 for both species. Permutation-importance
  tests identify `Abies_balsamea_distance_km` as the only conifer
  covariate with an FDR-q close to 0 for both species. Wilcoxon
  signed-rank tests show that the static-conifer sensitivity analysis
  gives systematically smaller northward gains than the environment-only
  projection (median difference ≈ 22 p.p. in area change for sugar, ≈ 35
  p.p. for red; ≈ 170 km and ≈ 145 km in latitudinal shift;
  all p < 0.01).

**Statistical caveat.** Supplementary 5 and 6 use only two spatial-block
folds given the 179 (sugar) and 217 (red) occupied cells, and the
permutation-importance test uses a small number of repeats per predictor
(see `feature_importance_*.csv` for the `permutation_std` column). The
paired bootstrap and McNemar test partly compensate by operating on the
pooled held-out predictions; still, we interpret the ROC-AUC and
permutation values primarily for ranking rather than for absolute
calibration. See `Forestry_methods_results_draft_FINAL.md` for the
matching Methods-and-Results text.

<!-- s7-master:v1 -->
## S7.8 Master cross-document statistical synthesis

The next cell loads `outputs/summary/stats/master_stats_synthesis.csv`,
a long-format table that consolidates **every** inferential output of
Supplementary 1–6 and links each row to the section of the main
manuscript (`Lizarazo et al_2026.docx`) where the same number is
discussed.

Columns:

| column | meaning |
|---|---|
| `supplementary` | source notebook (S1–S6) |
| `docx_section` | anchor in `Lizarazo et al_2026.docx` (line ranges of `_docx_text.txt`) |
| `family` | family of related tests (used for BH-FDR) |
| `species` | `sugar`, `red`, `ES vs EO`, or `both` |
| `test` / `statistic` | description and statistic name |
| `df` | reference / effective d.f. (NA for non-parametric tests) |
| `value` | point estimate of the statistic |
| `ci_low`, `ci_high` | 95 % CI when available |
| `p_value` | raw two-sided p-value |
| `p_bh_q` | BH-FDR within the family |
| `p_bh_q_global` | BH-FDR across the entire synthesis |
| `effect_size` / `effect_size_metric` | rank-biserial r, mean \|SHAP\|, median paired difference, … |
| `n` | sample size used by the test |
| `note` | short qualitative annotation |

**How to use it.** Sort by `docx_section` to read every supplementary
result that supports a given paragraph of the manuscript, or filter by
`family` to see all tests sharing a common FDR correction.


In [12]:
# <!-- s7-master:v1 -->
import pandas as pd
from pathlib import Path
MASTER = Path('outputs') / 'summary' / 'stats' / 'master_stats_synthesis.csv'
master = pd.read_csv(MASTER)
print(f'Rows: {len(master)} | Families: {master.family.nunique()} | DOCX anchors: {master.docx_section.nunique()}')
show_cols = ['supplementary','docx_section','family','species','test',
             'statistic','df','value','ci_low','ci_high',
             'p_value','p_bh_q','p_bh_q_global','effect_size','effect_size_metric','n']
master[show_cols].round(4)

Rows: 58 | Families: 7 | DOCX anchors: 4


,supplementary,docx_section,family,species,test,statistic,df,value,ci_low,ci_high,p_value,p_bh_q,p_bh_q_global,effect_size,effect_size_metric,n
0,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for elevation,|z| or chi-sq,NaN,92.9000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
1,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio1 (annual mean T),|z| or chi-sq,NaN,15.3000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
2,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio24 (radiation wettest quarter),|z| or chi-sq,NaN,11.9000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
3,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio34 (warm-season moisture),|z| or chi-sq,NaN,11.9000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
4,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio8 (T wettest quarter),|z| or chi-sq,NaN,11.4000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
5,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio30 (lowest moisture),|z| or chi-sq,NaN,11.1000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
6,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio31 (moisture seasonality),|z| or chi-sq,NaN,10.9000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
7,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio15 (precipitation seasonality),|z| or chi-sq,NaN,10.8000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
8,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,|z| / smooth chi-sq for bio27 (radiation coldest quarter),|z| or chi-sq,NaN,9.7000,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
9,S3,Methods §Modelling Spatial and Environmental Drivers (L24–L27); Results §Spatial drivers of dominance (L37–L42),GAM Wald (parametric + smooth),both,"|z| / smooth chi-sq for s(longitude, latitude) smooth",|z| or chi-sq,NaN,NaN,NaN,NaN,0.0000,NaN,0.0000,NaN,NaN,1413270.0
